# Partie I : Exploration, preprocessing, non supervisé

**Objectif de ce notebook** : charger le dataset *Myocardial infarction complications* et produire trois objets, utilisés ensuite par tout le groupe :

| Objet | Contenu |
|---|---|
| `y` | la cible : le patient est-il décédé (1) ou a-t-il survécu (0) ? |
| `X_admission` | les variables connues **à l'admission** à l'hôpital |
| `X_jour3` | les variables connues **au 3ᵉ jour** (toutes les variables d'entrée) |

**Pourquoi deux jeux de variables ?** On veut savoir si l'on peut prédire les complications dès l'arrivée du patient, puis si les informations recueillies pendant 3 jours améliorent la prédiction. Utiliser au jour 0 une information qui n'existe qu'au jour 3 serait de la **fuite de données** (*data leakage*) : le modèle paraîtrait excellent mais serait inutilisable en pratique.

Le dataset contient 1700 patients, 111 variables d'entrée (*features*) et 12 complications possibles.

## 1. Import des bibliothèques

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

## 2. Chargement du fichier CSV

Le fichier doit se trouver à côté du notebook ou dans un dossier `data/`. La cellule ci-dessous cherche à plusieurs endroits et affiche un message clair si elle ne trouve rien (c'était la cause du `FileNotFoundError`).

On utilise `index_col="ID"` : la colonne `ID` identifie le patient, ce n'est pas une variable à donner au modèle.

In [3]:
NAME_FILES = "Myocardial_infarction_complications_Database.csv"
PATH_FILES = [NAME_FILES, os.path.join("data", NAME_FILES), os.path.join("..", "data", NAME_FILES)]

chemin = next((c for c in PATH_FILES if os.path.exists(c)), None)
if chemin is None:
    raise FileNotFoundError(
        f"{NAME_FILES} introuvable. Cherché dans : {PATH_FILES}. "
        "Copiez le CSV dans le dossier du notebook ou dans data/."
    )

df = pd.read_csv(chemin, index_col="ID")
print("Fichier chargé :", chemin)
print("Dimensions :", df.shape)   # attendu : (1700, 123) = 111 entrées + 12 complications

Fichier chargé : data/Myocardial_infarction_complications_Database.csv
Dimensions : (1700, 123)


Les valeurs manquantes sont des **cases vides** dans le CSV, que pandas lit directement comme `NaN` (pas besoin de `na_values="?"`).

## 3. Séparer les variables d'entrée et les complications

Les 12 dernières colonnes sont des **complications** (ce qu'on veut prédire). Toutes les autres sont des variables d'entrée.

On les sélectionne **par leur nom** et non par leur position (`iloc[:, 2:113]`). Avec un numéro, un décalage d'une colonne passe inaperçu et peut mettre une complication dans `X`, ce qui donne une fuite de données.

In [ ]:
COMPLICATIONS = [
    "FIBR_PREDS",  # fibrillation auriculaire
    "PREDS_TAH",   # tachycardie supraventriculaire
    "JELUD_TAH",   # tachycardie ventriculaire
    "FIBR_JELUD",  # fibrillation ventriculaire
    "A_V_BLOK",    # bloc auriculo-ventriculaire du 3e degré
    "OTEK_LANC",   # œdème pulmonaire
    "RAZRIV",      # rupture myocardique
    "DRESSLER",    # syndrome de Dressler
    "ZSN",         # insuffisance cardiaque chronique
    "REC_IM",      # récidive d'infarctus
    "P_IM_STEN",   # angine post-infarctus
    "LET_IS",      # issue létale : 0 = survie, 1 à 7 = cause du décès
]

# Vérification : toutes ces colonnes existent bien dans le fichier
manquantes = [c for c in COMPLICATIONS if c not in df.columns]
assert not manquantes, f"Colonnes absentes du fichier : {manquantes}"

X_toutes = df.drop(columns=COMPLICATIONS)   # 111 variables d'entrée
print("Variables d'entrée :", X_toutes.shape[1])
print("Complications      :", len(COMPLICATIONS))
assert X_toutes.shape[1] == 111, "On attendait 111 variables d'entrée"

## 4. La cible `y` : décès ou survie

`LET_IS` vaut 0 si le patient survit, et de 1 à 7 selon la cause du décès (1 choc cardiogénique, 2 œdème pulmonaire, 3 rupture myocardique, 4 aggravation de l'insuffisance cardiaque, 5 thromboembolie, 6 asystolie, 7 fibrillation ventriculaire).

On la **binarise** : 0 = survie, 1 = décès (quelle qu'en soit la cause). On obtient un problème de classification binaire.

In [ ]:
y = (df["LET_IS"] > 0).astype(int)
y.name = "deces"

print("Répartition de la cible :")
print(y.value_counts().rename({0: "survie", 1: "décès"}))
print(f"\nProportion de décès : {y.mean():.1%}")

y.value_counts().rename({0: "survie", 1: "décès"}).plot(kind="bar", color=["steelblue", "indianred"])
plt.title("Répartition survie / décès")
plt.ylabel("Nombre de patients")
plt.xticks(rotation=0)
plt.show()

**À retenir** : les classes sont **déséquilibrées** (les décès sont une minorité). L'*accuracy* ne suffira donc pas : un modèle qui prédit toujours « survie » aurait un score élevé tout en étant inutile. On utilisera le recall, la PR-AUC, le F1 et la *balanced accuracy*, avec des splits **stratifiés**.

## 5. Quelles variables sont disponibles à quel moment ?

Les colonnes suivent l'ordre chronologique du dossier médical. Certaines ne sont connues que **plus tard** dans l'hospitalisation :

| Groupe | Colonnes | Moment où l'information existe |
|---|---|---|
| Antécédents, ECG, biologie, délai | `AGE` … `TIME_B_S` | à l'admission |
| Traitements des urgences | `NA_KB`, `NOT_NA_KB`, `LID_KB`, `NITR_S` | à l'admission (équipe d'urgence) |
| Douleur récurrente | `R_AB_1_n`, `R_AB_2_n`, `R_AB_3_n` | `_1_` premières heures, `_2_` jour 2, `_3_` jour 3 |
| Opioïdes en soins intensifs | `NA_R_1_n`, `NA_R_2_n`, `NA_R_3_n` | idem |
| Anti-inflammatoires en soins intensifs | `NOT_NA_1_n`, `NOT_NA_2_n`, `NOT_NA_3_n` | idem |
| Autres traitements en soins intensifs | `LID_S_n`, `B_BLOK_S_n`, `ANT_CA_S_n`, `GEPAR_S_n`, `ASP_S_n`, `TIKL_S_n`, `TRENT_S_n` | pendant le séjour (moment exact à vérifier) |

⚠️ **Cette liste est une décision à valider en groupe.** L'article fourni (`2007_03788v1.pdf`) ne la donne pas : elle figure dans la description du dataset sur UCI. Les neuf colonnes ci-dessous (`R_AB_*`, `NA_R_*`, `NOT_NA_*`) correspondent au choix de votre camarade, retraduit en noms de colonnes. Le réglage `STRICT_ADMISSION` permet d'être plus prudent.

In [ ]:
# Colonnes dont la valeur dépend de ce qui se passe APRÈS l'arrivée (jours 1 à 3)
COLONNES_APRES_ADMISSION = [
    "R_AB_1_n", "R_AB_2_n", "R_AB_3_n",          # récidive de la douleur
    "NA_R_1_n", "NA_R_2_n", "NA_R_3_n",          # opioïdes en soins intensifs
    "NOT_NA_1_n", "NOT_NA_2_n", "NOT_NA_3_n",    # anti-inflammatoires en soins intensifs
]

# Traitements en soins intensifs dont le moment exact est incertain.
# Si STRICT_ADMISSION = True, on les retire aussi du scénario « admission ».
COLONNES_SOINS_INTENSIFS_INCERTAINES = [
    "LID_S_n", "B_BLOK_S_n", "ANT_CA_S_n", "GEPAR_S_n", "ASP_S_n", "TIKL_S_n", "TRENT_S_n",
]
STRICT_ADMISSION = False   # à décider en groupe

a_retirer = list(COLONNES_APRES_ADMISSION)
if STRICT_ADMISSION:
    a_retirer += COLONNES_SOINS_INTENSIFS_INCERTAINES

absentes = [c for c in a_retirer if c not in X_toutes.columns]
assert not absentes, f"Colonnes absentes : {absentes}"

## 6. Construction de `X_admission` et `X_jour3`

- `X_jour3` : toutes les variables d'entrée (information complète au 3ᵉ jour).
- `X_admission` : les mêmes, moins les colonnes de la cellule précédente.

On garde des **DataFrames** (avec les noms de colonnes) et on ne les convertit pas en tableaux numpy : sans les noms, on ne peut plus interpréter les résultats (importance des variables, SHAP, arbre de décision).

In [ ]:
X_jour3 = X_toutes.copy()
X_admission = X_toutes.drop(columns=a_retirer)

print("X_jour3     :", X_jour3.shape)
print("X_admission :", X_admission.shape)
print("Colonnes retirées pour l'admission :", len(a_retirer))

## 7. Vérifications de sécurité

Ces `assert` échouent si une erreur s'est glissée : mieux vaut un blocage ici qu'un résultat faux découvert le jour de la présentation.

In [ ]:
# 1. Aucune complication (cible) ne doit être dans les variables d'entrée
assert not set(COMPLICATIONS) & set(X_jour3.columns), "Fuite : une complication est dans X_jour3"
assert not set(COMPLICATIONS) & set(X_admission.columns), "Fuite : une complication est dans X_admission"

# 2. X_admission est bien un sous-ensemble de X_jour3
assert set(X_admission.columns) <= set(X_jour3.columns)

# 3. Mêmes patients, dans le même ordre, pour X et y
assert X_admission.index.equals(y.index) and X_jour3.index.equals(y.index)

# 4. Aucune colonne retirée n'est restée dans X_admission
assert not set(a_retirer) & set(X_admission.columns)

print("Toutes les vérifications sont passées.")

## 8. Coup d'œil aux valeurs manquantes

Le dataset contient beaucoup de valeurs manquantes. Une valeur manquante signifie souvent qu'**un examen n'a pas été réalisé**, pas forcément une erreur de saisie. C'est un point à discuter pour l'imputation.

In [ ]:
pct_manquant = X_jour3.isna().mean().sort_values(ascending=False) * 100
print(pct_manquant.head(10).round(1).to_string())

pct_manquant.head(15).sort_values().plot(kind="barh", figsize=(6, 5), color="darkorange")
plt.xlabel("% de valeurs manquantes")
plt.title("Les 15 colonnes les plus incomplètes")
plt.tight_layout()
plt.show()

## 9. Séparation train / test (stratifiée)

On sépare **une seule fois** les patients, avec les mêmes indices pour les deux scénarios (admission et jour 3). Ainsi, la comparaison entre les deux se fait sur exactement les mêmes patients de test.

- `stratify=y` conserve la proportion de décès dans le train et dans le test.
- `random_state=42` rend le découpage reproductible : tout le groupe obtient les mêmes résultats.

⚠️ On ne fait **aucune imputation ni standardisation ici** : ces opérations doivent être apprises sur le train uniquement, à l'intérieur d'un `Pipeline` scikit-learn. Sinon, des informations du test fuiraient dans l'entraînement.

In [ ]:
SEED = 42

idx_train, idx_test = train_test_split(
    y.index, test_size=0.2, stratify=y, random_state=SEED
)

X_adm_train, X_adm_test = X_admission.loc[idx_train], X_admission.loc[idx_test]
X_j3_train,  X_j3_test  = X_jour3.loc[idx_train],     X_jour3.loc[idx_test]
y_train, y_test = y.loc[idx_train], y.loc[idx_test]

print(f"Train : {len(idx_train)} patients, {y_train.mean():.1%} de décès")
print(f"Test  : {len(idx_test)} patients, {y_test.mean():.1%} de décès")

## Récapitulatif

| Objet | Description |
|---|---|
| `y` | cible binaire décès / survie |
| `X_admission` | variables connues à l'admission |
| `X_jour3` | toutes les variables d'entrée |
| `X_adm_train`, `X_adm_test`, `X_j3_train`, `X_j3_test`, `y_train`, `y_test` | découpage commun du groupe |

**Étapes suivantes** : exploration (distributions, corrélations), puis preprocessing dans un `Pipeline` (imputation, standardisation), puis les modèles.